In [4]:
import os
import re
import time
import requests
import pandas as pd
from io import StringIO
from pathlib import Path
from bs4 import BeautifulSoup
from tqdm import tqdm
from joblib import Parallel, delayed
from sec_edgar_downloader import Downloader
import yfinance as yf
import numpy as np
import warnings

# --- SILENCE THE WARNINGS ---
warnings.filterwarnings('ignore', category=FutureWarning)

# --- CONFIGURATION ---
USER_NAME = "Michael Aoun"
USER_EMAIL = "michael.aoun@ieseg.fr"
DOWNLOAD_FOLDER = "SEC_Data"

In [5]:
print("Fetching S&P 500 tickers from Wikipedia...")
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
response = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(response.text))
sp500_tickers = tables[0]['Symbol'].tolist()

print(f"✅ Successfully grabbed {len(sp500_tickers)} tickers.")
print(f"First 5: {sp500_tickers[:5]}")

Fetching S&P 500 tickers from Wikipedia...
✅ Successfully grabbed 503 tickers.
First 5: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN']


In [3]:
dl = Downloader(USER_NAME, USER_EMAIL, DOWNLOAD_FOLDER)
failed_tickers = []

print("Starting download for 10-Ks from 2005 through 2025 (excluding 2026)...")

for ticker in tqdm(sp500_tickers, desc="Downloading 10-Ks"):
    try:
        # download_details=True gets the HTML files
        # Using 'after' and 'before' explicitly filters the filing dates
        dl.get("10-K", ticker, after="2005-01-01", before="2025-12-31", download_details=True)
        
        # --- THE TXT CLEANUP ---
        # Find and delete every .txt file in this ticker's newly downloaded folder
        ticker_dir = Path(DOWNLOAD_FOLDER) / "sec-edgar-filings" / ticker / "10-K"
        if ticker_dir.exists():
            for txt_file in ticker_dir.rglob("*.txt"):
                txt_file.unlink() 
        
        # SEC speed limit compliance
        time.sleep(0.2)
        
    except Exception as e:
        failed_tickers.append((ticker, str(e)))

print("\n✅ Download phase complete!")
if failed_tickers:
    print(f"⚠️ Could not download data for {len(failed_tickers)} tickers.")

Starting download for 10-Ks from 2005 through 2025 (excluding 2026)...



✅ Download phase complete!
⚠️ Could not download data for 2 tickers.


In [6]:
def get_year_from_folder(folder_name):
    """Extracts the filing year from the SEC accession number."""
    try:
        parts = folder_name.split('-')
        if len(parts) >= 2:
            yy = int(parts[1])
            return 1900 + yy if yy >= 50 else 2000 + yy
    except ValueError:
        pass
    return "Unknown"

def extract_item_1a(file_path):
    """Parses the 10-K HTML file, extracting Item 1A while destroying HTML bloat."""
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
            raw_html = file.read()
            
        soup = BeautifulSoup(raw_html, "lxml")
        
        # Instantly delete tables, images, and graphics to save CPU memory
        for bloated_tag in soup(['table', 'img', 'svg']):
            bloated_tag.decompose() 
            
        text = soup.get_text(separator=' ')
        text = re.sub(r'\s+', ' ', text)
        
        start_regex = re.compile(r'item\s+1a\.?\s*(?:risk\s+factors)?', re.IGNORECASE)
        end_regex = re.compile(r'item\s+(?:1b|2)\.?\s*(?:unresolved\s+staff\s+comments|properties)?', re.IGNORECASE)
        
        starts = list(start_regex.finditer(text))
        ends = list(end_regex.finditer(text))
        
        if not starts: return "Flagged: 'Item 1A' Start Marker Not Found", ""
        if not ends: return "Flagged: 'Item 1B/2' End Marker Not Found", ""

        for start_match in reversed(starts):
            start_idx = start_match.end()
            valid_ends = [e.start() for e in ends if e.start() > start_idx]
            
            if valid_ends:
                end_idx = valid_ends[0]
                chunk = text[start_idx:end_idx].strip()
                
                # Minimum length check remains to avoid grabbing the Table of Contents
                if len(chunk) < 1500: continue
                else: return "Success", chunk
                    
        return "Flagged: All matches too short (Likely caught in TOC)", ""

    except Exception as e:
        return f"Flagged: System Error - {str(e)}", ""

def process_single_file(args):
    """Unpacks arguments and runs extraction."""
    ticker, year, file_path = args
    
    # 10 MB File size limit has been removed
        
    status, extracted_text = extract_item_1a(file_path)
    
    return {
        "Ticker": ticker, "Year": year,
        "Extraction_Status": status, "Text": extracted_text
    }

In [7]:
base_path = Path(DOWNLOAD_FOLDER) / "sec-edgar-filings"
tasks = []

if base_path.exists():
    for ticker_dir in base_path.iterdir():
        if ticker_dir.is_dir():
            ticker = ticker_dir.name
            ten_k_dir = ticker_dir / "10-K"
            
            if ten_k_dir.exists():
                for filing_dir in ten_k_dir.iterdir():
                    if filing_dir.is_dir():
                        year = get_year_from_folder(filing_dir.name)
                        # Locate the primary HTML document
                        html_files = list(filing_dir.glob("primary-document.htm*"))
                        
                        if html_files:
                            file_path = html_files[0]
                            tasks.append((ticker, year, file_path))

print(f"Gathered {len(tasks)} HTML files to process.")
print("Spinning up CPU cores for high-speed extraction...")

# Extract in Parallel
results_generator = Parallel(n_jobs=-1, return_as="generator")(
    delayed(process_single_file)(task) for task in tasks
)
results = list(tqdm(results_generator, total=len(tasks), desc="Extracting Item 1A"))

# Build Final Dataset
df = pd.DataFrame(results)
df = df.sort_values(by=["Ticker", "Year"]).reset_index(drop=True)

# Add Character Count
df['Character_Count'] = df['Text'].str.len()

print("\nExtraction Complete!")
print("\nStatus Breakdown:")
print(df['Extraction_Status'].value_counts().head(10))

# Keep only the rows where the Year is NOT 2005
df = df[df['Year'] != 2005]

# Optional: Reset the index after dropping rows
df = df.reset_index(drop=True)

# Drop any rows where the 'Text' column is empty/null
df = df.dropna(subset=['Text'])

# Reset the index again to keep your row numbers perfectly sequential
df = df.reset_index(drop=True)

Gathered 9165 HTML files to process.
Spinning up CPU cores for high-speed extraction...


Extracting Item 1A: 100%|██████████| 9165/9165 [33:48<00:00,  4.52it/s]  


Extraction Complete!

Status Breakdown:
Extraction_Status
Success                                                  6479
Flagged: 'Item 1B/2' End Marker Not Found                1444
Flagged: 'Item 1A' Start Marker Not Found                1074
Flagged: All matches too short (Likely caught in TOC)     168
Name: count, dtype: int64


In [ ]:
# Create empty lists to store our new target variables
returns = []
target_labels = []

print("Fetching historical stock data to build target labels...")

# Loop through each row in your existing df
for index, row in tqdm(df.iterrows(), total=len(df), desc="Calculating Returns"):
    ticker = row['Ticker']
    year = row['Year']
    
    try:
        # Define our "Forward Year" trading window (April 1st to March 31st)
        start_date = f"{year}-04-01"
        end_date = f"{year + 1}-03-31"
        
        # Fetch the historical data quietly
        stock = yf.download(ticker, start=start_date, end=end_date, progress=False)
        
        # Check if we actually got data back
        if len(stock) > 2:
            # We already use the .iloc[0] method they recommend, but wrapping it in float() 
            # triggers a pandas warning. Using .item() is the cleanest way in modern pandas.
            start_price = stock['Close'].iloc[0].item()
            end_price = stock['Close'].iloc[-1].item()
            
            # Calculate the percentage return
            pct_return = (end_price - start_price) / start_price
            
            # Create the Binary Label: 1 if positive return, 0 if negative or flat
            label = 1 if pct_return > 0 else 0
            
            returns.append(pct_return)
            target_labels.append(label)
        else:
            returns.append(np.nan)
            target_labels.append(np.nan)
            
    except Exception as e:
        returns.append(np.nan)
        target_labels.append(np.nan)

# Attach the new columns to your dataframe
df['Forward_1Y_Return'] = returns
df['Target_Up_Down'] = target_labels

# Drop the rows where yfinance couldn't find historical price data
df = df.dropna(subset=['Target_Up_Down']).reset_index(drop=True)

# Convert the binary label to an integer
df['Target_Up_Down'] = df['Target_Up_Down'].astype(int)

print("\n✅ Label Generation Complete!")
print("\nTarget Label Distribution (1 = Up, 0 = Down):")
print(df['Target_Up_Down'].value_counts(normalize=True))

# Save to CSV
df.to_csv(r'../data/SP500_RiskFactors_Dataset.csv', index=False, escapechar='\\')

print(f"\n✅ Data successfully saved to {'../data/SP500_RiskFactors_Dataset.csv'}")

Fetching historical stock data to build target labels...


Calculating Returns: 100%|██████████| 8792/8792 [24:35<00:00,  5.96it/s]  



✅ Label Generation Complete!

Target Label Distribution (1 = Up, 0 = Down):
Target_Up_Down
1    0.678571
0    0.321429
Name: proportion, dtype: float64

✅ Data successfully saved to ../data/SP500_RiskFactors_Dataset.csv
